# Time Series Fundamentals

[← Back to lesson](https://ml-viz.vercel.app/courses/time-series/01-time-series-fundamentals)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
np.random.seed(42)

## Generating synthetic series with 4 components

In [ ]:
t = np.arange(48)
trend = 100 + 2.1 * t
seasonal = 25 * np.sin(2 * np.pi * t / 12)
residual = np.random.normal(0, 7, 48)
observed = trend + seasonal + residual

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for ax, y, title in zip(axes, [observed, trend, seasonal, residual],
                         ['Observed', 'Trend', 'Seasonal', 'Residual']):
    ax.plot(t, y, color='#2dd4bf', linewidth=1.5)
    ax.set_title(title, color='white', fontsize=11)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Stationarity: ADF test

In [ ]:
try:
    from statsmodels.tsa.stattools import adfuller
    raw_result = adfuller(observed)
    print(f"Raw series  — ADF stat: {raw_result[0]:.3f}, p-value: {raw_result[1]:.4f}")
    
    log_diff = np.diff(np.log(observed + 1e-9))
    diff_result = adfuller(log_diff)
    print(f"Log+diff    — ADF stat: {diff_result[0]:.3f}, p-value: {diff_result[1]:.4f}")
    print("Stationary after transformation!" if diff_result[1] < 0.05 else "Still non-stationary")
except ImportError:
    print("statsmodels not installed — run: pip install statsmodels")
    # Manual check: compute rolling mean/variance
    half = len(observed) // 2
    print(f"Mean first half: {observed[:half].mean():.1f}, second half: {observed[half:].mean():.1f}")
    print(f"Std first half:  {observed[:half].std():.1f},  second half: {observed[half:].std():.1f}")

## Computing sample ACF from scratch

In [ ]:
def sample_acf(y, max_lag=16):
    y_centered = y - y.mean()
    c0 = np.dot(y_centered, y_centered)
    return np.array([np.dot(y_centered[:len(y)-k], y_centered[k:]) / c0
                     for k in range(1, max_lag + 1)])

acf_vals = sample_acf(observed)
lags = np.arange(1, 17)
sig_band = 1.96 / np.sqrt(len(observed))

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(lags, acf_vals, color='#818cf8', alpha=0.8)
ax.axhline(sig_band, color='#f59e0b', linestyle='--', linewidth=1, label='±95% band')
ax.axhline(-sig_band, color='#f59e0b', linestyle='--', linewidth=1)
ax.set_xlabel('Lag', color='white')
ax.set_ylabel('ACF', color='white')
ax.set_title('Sample Autocorrelation Function', color='white')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## ✏️ Your turn: Generate an AR(1) series

In [ ]:
# TODO(you): Generate an AR(1) series with phi=0.5 and n=100
# y[t] = 0.5 * y[t-1] + noise, noise ~ N(0, 1)

# YOUR CODE HERE
y_ar1 = None  # replace

assert y_ar1 is not None, "Assign y_ar1"
assert len(y_ar1) == 100, "Should have 100 observations"
print("✓ AR(1) series generated")

<details><summary>Solution</summary>

```python
rng = np.random.default_rng(42)
n = 100
y_ar1 = np.zeros(n)
for t in range(1, n):
    y_ar1[t] = 0.5 * y_ar1[t-1] + rng.normal(0, 1)
```
</details>